In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

master_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/Project_Root/Merged_Dataset/master_df.xlsx"
)

Data Profiling

In [5]:
print("Shape:", master_df.shape)

print("\n--- Data Types ---")
print(master_df.dtypes)

print("\n--- Missing Values ---")
print(master_df.isnull().sum())

print("\n--- Unique Values ---")
print(master_df.nunique())

print("\n--- Duplicate Rows ---")
print(master_df.duplicated().sum())

print("\n--- Statistical Summary ---")
print(master_df.describe())

Shape: (7043, 56)

--- Data Types ---
Customer ID                           object
Gender                                object
Age                                    int64
Under 30                              object
Senior Citizen                        object
Married                               object
Dependents                            object
Number of Dependents                   int64
Service ID                            object
Quarter_x                             object
Referred a Friend                     object
Number of Referrals                    int64
Tenure in Months                       int64
Offer                                 object
Phone Service                         object
Avg Monthly Long Distance Charges    float64
Multiple Lines                        object
Internet Service                      object
Internet Type                         object
Avg Monthly GB Download                int64
Online Security                       object
Online Backup    

In [6]:
profile = pd.DataFrame({
    "Data Type": master_df.dtypes,
    "Missing Values": master_df.isnull().sum(),
    "Missing %": (master_df.isnull().sum() / len(master_df) * 100).round(2),
    "Unique Values": master_df.nunique()
})

profile

,Data Type,Missing Values,Missing %,Unique Values
Customer ID,object,0,0.00,7043
Gender,object,0,0.00,2
Age,int64,0,0.00,62
Under 30,object,0,0.00,2
Senior Citizen,object,0,0.00,2
Married,object,0,0.00,2
Dependents,object,0,0.00,2
Number of Dependents,int64,0,0.00,12
Service ID,object,0,0.00,7043
Quarter_x,object,0,0.00,1


##**Data Cleaning & Preprocessing**


###Understanding Data


In [7]:
print("Dataset Shape\n")
print(master_df.shape)

print("\n Dataset Information\n")
master_df.info()

# print("\n Head\n")
# master_df.head()

Dataset Shape

(7043, 56)

 Dataset Information

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 56 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Under 30                           7043 non-null   object 
 4   Senior Citizen                     7043 non-null   object 
 5   Married                            7043 non-null   object 
 6   Dependents                         7043 non-null   object 
 7   Number of Dependents               7043 non-null   int64  
 8   Service ID                         7043 non-null   object 
 9   Quarter_x                          7043 non-null   object 
 10  Referred a Friend                  7043 non-null   object 
 11  Number 

*   7043 rows × 56 columns
*   No missing values in most columns.
*   Numeric columns have correct data types.

**Things to address**

1. Offer → 3166 non-null (3877 missing)
2. Internet Type → 5517 non-null (1526 missing)
3. Churn Reason → 1869 non-null (5174 missing)
4. Quarter_x and Quarter_y (probably duplicate information)
5. Several ID columns (Customer ID, Service ID, Status ID, Location ID, ID) that likely won't be used for model training.

####**Check for Duplicates**


In [8]:

duplicate_rows = master_df.duplicated().sum()
print("Duplicate Rows:", duplicate_rows)

Duplicate Rows: 0


####**Count and calculate percentage of missing values**

In [9]:
missing = pd.DataFrame({
    "Missing Values": master_df.isnull().sum(),
    "Percentage": (master_df.isnull().sum() / len(master_df)) * 100
})

missing = missing[missing["Missing Values"] > 0]
missing.sort_values(by="Missing Values", ascending=False)

,Missing Values,Percentage
Churn Reason,5174,73.463013
Offer,3877,55.047565
Internet Type,1526,21.666903


In [10]:
# We're investigating whether missing Internet Type is related to:
# Internet Service = No

# It is missing because the customer doesn't have internet service.

master_df[master_df["Internet Type"].isnull()]["Internet Service"].value_counts()

# Check the Internet Service values for customers with missing Internet Type.
# All 1,526 missing Internet Type values belong to customers with no Internet Service.

,count
Internet Service,
No,1526


In [11]:
master_df[master_df["Offer"].isnull()]["Customer Status"].value_counts()

,count
Customer Status,
Stayed,2547
Churned,1051
Joined,279


In [12]:
# Among customers with a missing Offer, what is their Customer Status?
# Among customers whose Offer is missing, how many are Stayed, Churned, or Joined.
master_df[master_df["Offer"].isnull()]["Customer Status"].value_counts()

,count
Customer Status,
Stayed,2547
Churned,1051
Joined,279


In [13]:
# check whether missing Offer is related to Churn Label.
master_df[master_df["Offer"].isnull()]["Churn Label"].value_counts()

,count
Churn Label,
No,2826
Yes,1051


In [14]:
# which Offer values exist for customers who actually have an offer.
master_df["Offer"].value_counts(dropna=False)

,count
Offer,
NaN,3877
Offer B,824
Offer E,805
Offer D,602
Offer A,520
Offer C,415


In [15]:
master_df[master_df["Offer"].isnull()]["Referred a Friend"].value_counts()

,count
Referred a Friend,
No,2119
Yes,1758


In [16]:
master_df[master_df["Offer"].isnull()]["Tenure in Months"].describe()

,Tenure in Months
count,3877.000000
mean,31.566159
std,24.052436
min,1.000000
25%,8.000000
50%,29.000000
75%,53.000000
max,72.000000


In [17]:
master_df[master_df["Offer"].isnull()]["Contract"].value_counts()

,count
Contract,
Month-to-Month,2020
Two Year,1015
One Year,842


In [18]:
master_df[master_df["Offer"].isnull()]["Internet Service"].value_counts()

,count
Internet Service,
Yes,3024
No,853


In [19]:
pd.crosstab(
    master_df["Customer Status"],
    master_df["Offer"],
    dropna=False
)

Offer,Offer A,Offer B,Offer C,Offer D,Offer E,NaN
Customer Status,,,,,,
Churned,35,101,95,161,426,1051
Joined,0,0,0,0,175,279
Stayed,485,723,320,441,204,2547


In [20]:
master_df[master_df["Churn Reason"].isnull()]["Customer Status"].value_counts()

,count
Customer Status,
Stayed,4720
Joined,454


In [21]:
master_df["Internet Type"] = master_df["Internet Type"].fillna("No Internet")
master_df["Internet Type"].isnull().sum()

np.int64(0)

In [22]:
master_df["Offer"] = master_df["Offer"].fillna("No Offer")
master_df["Offer"].isnull().sum()

np.int64(0)

In [23]:
master_df[master_df["Churn Reason"].isnull()]["Customer Status"].value_counts()

,count
Customer Status,
Stayed,4720
Joined,454


In [24]:
master_df["Churn Reason"] = master_df["Churn Reason"].fillna("Not Churned")
master_df["Churn Reason"].isnull().sum()

np.int64(0)

In [25]:
master_df.columns.tolist()

['Customer ID',
 'Gender',
 'Age',
 'Under 30',
 'Senior Citizen',
 'Married',
 'Dependents',
 'Number of Dependents',
 'Service ID',
 'Quarter_x',
 'Referred a Friend',
 'Number of Referrals',
 'Tenure in Months',
 'Offer',
 'Phone Service',
 'Avg Monthly Long Distance Charges',
 'Multiple Lines',
 'Internet Service',
 'Internet Type',
 'Avg Monthly GB Download',
 'Online Security',
 'Online Backup',
 'Device Protection Plan',
 'Premium Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Streaming Music',
 'Unlimited Data',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charge',
 'Total Charges',
 'Total Refunds',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Revenue',
 'Status ID',
 'Quarter_y',
 'Satisfaction Score',
 'Customer Status',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason',
 'Location ID',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'ID',
 'Population']

In [26]:
master_df["Quarter_x"].equals(master_df["Quarter_y"])

True

In [27]:
master_df = master_df.drop(columns=["Quarter_y"])

In [28]:
print(master_df.columns.tolist())

['Customer ID', 'Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Service ID', 'Quarter_x', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Status ID', 'Satisfaction Score', 'Customer Status', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason', 'Location ID', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'ID', 'Population']


In [29]:

master_df = master_df.drop(columns=[
    "Customer ID",
    "Service ID",
    "Status ID",
    "Location ID",
    "ID"
])

In [30]:
print(master_df.shape)

(7043, 50)


In [31]:
constant_cols = [
    col for col in master_df.columns
    if master_df[col].nunique() <= 1
]

print(constant_cols)

['Quarter_x', 'Country', 'State']


In [32]:

master_df = master_df.drop(columns=[
    "Quarter_x",
    "Country",
    "State"
])

In [33]:
master_df.isnull().sum()[master_df.isnull().sum() > 0]

,0


In [34]:
master_df.dtypes.value_counts()

,count
object,27
int64,12
float64,8


In [35]:
for col in master_df.select_dtypes(include="object").columns:
    print("\n", col)
    print(master_df[col].unique())


 Gender
['Male' 'Female']

 Under 30
['No' 'Yes']

 Senior Citizen
['Yes' 'No']

 Married
['No' 'Yes']

 Dependents
['No' 'Yes']

 Referred a Friend
['No' 'Yes']

 Offer
['No Offer' 'Offer E' 'Offer D' 'Offer C' 'Offer B' 'Offer A']

 Phone Service
['No' 'Yes']

 Multiple Lines
['No' 'Yes']

 Internet Service
['Yes' 'No']

 Internet Type
['DSL' 'Fiber Optic' 'Cable' 'No Internet']

 Online Security
['No' 'Yes']

 Online Backup
['No' 'Yes']

 Device Protection Plan
['Yes' 'No']

 Premium Tech Support
['No' 'Yes']

 Streaming TV
['No' 'Yes']

 Streaming Movies
['Yes' 'No']

 Streaming Music
['No' 'Yes']

 Unlimited Data
['No' 'Yes']

 Contract
['Month-to-Month' 'One Year' 'Two Year']

 Paperless Billing
['Yes' 'No']

 Payment Method
['Bank Withdrawal' 'Credit Card' 'Mailed Check']

 Customer Status
['Churned' 'Stayed' 'Joined']

 Churn Label
['Yes' 'No']

 Churn Reason
['Competitor offered more data' 'Competitor made better offer'
 'Limited range of services' 'Extra data charges'
 'Comp

####**Clean Churn Reason**

In [36]:
master_df["Churn Reason"] = master_df["Churn Reason"].str.strip()

In [37]:
master_df["Churn Reason"].unique()


array(['Competitor offered more data', 'Competitor made better offer',
       'Limited range of services', 'Extra data charges',
       'Competitor had better devices', "Don't know",
       'Service dissatisfaction',
       'Lack of affordable download/upload speed',
       'Product dissatisfaction', 'Long distance charges',
       'Poor expertise of online support', 'Attitude of support person',
       'Network reliability', 'Competitor offered higher download speeds',
       'Moved', 'Price too high', 'Attitude of servicé provider',
       'Competitor made better offer @', 'Attitude of service provider',
       'Poor expertise of phone support', 'Deceased',
       '& Competitor made better offer', 'Compètitor made better offer',
       'Attitudè of servicé provider', 'Attitudè of service provider',
       'Not Churned', 'Long distance charges$',
       'Competitr made better offer', 'Price too hìgh',
       'Long dístance charges', 'Lack of self-service on Website',
       'Lack of s

In [38]:
replacements = {
    "Attitude of servicé provider": "Attitude of service provider",
    "Attitudè of servicé provider": "Attitude of service provider",
    "Attitudè of service provider": "Attitude of service provider",
    "Competitor made better offer @": "Competitor made better offer",
    "& Competitor made better offer": "Competitor made better offer",
    "Compètitor made better offer": "Competitor made better offer",
    "Competitr made better offer": "Competitor made better offer",
    "Long distance charges$": "Long distance charges",
    "Long dístance charges": "Long distance charges",
    "Price too hìgh": "Price too high",
    "Lack of self-service on Website #": "Lack of self-service on Website"
}

master_df["Churn Reason"] = master_df["Churn Reason"].replace(replacements)

In [39]:
master_df["Churn Reason"].unique()

array(['Competitor offered more data', 'Competitor made better offer',
       'Limited range of services', 'Extra data charges',
       'Competitor had better devices', "Don't know",
       'Service dissatisfaction',
       'Lack of affordable download/upload speed',
       'Product dissatisfaction', 'Long distance charges',
       'Poor expertise of online support', 'Attitude of support person',
       'Network reliability', 'Competitor offered higher download speeds',
       'Moved', 'Price too high', 'Attitude of service provider',
       'Poor expertise of phone support', 'Deceased', 'Not Churned',
       'Lack of self-service on Website'], dtype=object)

In [40]:

master_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,7043.0,46.509726,16.750352,19.000000,32.000000,46.000000,60.000000,80.000000
Number of Dependents,7043.0,0.468124,0.963521,-2.000000,0.000000,0.000000,0.000000,9.000000
Number of Referrals,7043.0,1.951867,3.001199,0.000000,0.000000,0.000000,3.000000,11.000000
Tenure in Months,7043.0,32.386767,24.542061,1.000000,9.000000,29.000000,55.000000,72.000000
Avg Monthly Long Distance Charges,7043.0,22.958954,15.448113,0.000000,9.210000,22.890000,36.395000,49.990000
Avg Monthly GB Download,7043.0,20.515405,20.418940,0.000000,3.000000,17.000000,27.000000,85.000000
Monthly Charge,7043.0,64.761692,30.090047,18.250000,35.500000,70.350000,89.850000,118.750000
Total Charges,7043.0,2280.381264,2266.220462,18.800000,400.150000,1394.550000,3786.600000,8684.800000
Total Refunds,7043.0,1.962182,7.902614,0.000000,0.000000,0.000000,0.000000,49.790000
Total Extra Data Charges,7043.0,6.860713,25.104978,0.000000,0.000000,0.000000,0.000000,150.000000


In [41]:
master_df[master_df["Number of Dependents"] < 0][["Number of Dependents"]]

,Number of Dependents
18,-1
32,-1
122,-2


In [42]:
master_df.loc[
    master_df["Number of Dependents"] < 0,
    "Number of Dependents"
] = 0

In [43]:
master_df["Number of Dependents"].min()

0

In [44]:
numeric_cols = master_df.select_dtypes(include="number").columns

master_df[numeric_cols].lt(0).sum()

,0
Age,0
Number of Dependents,0
Number of Referrals,0
Tenure in Months,0
Avg Monthly Long Distance Charges,0
Avg Monthly GB Download,0
Monthly Charge,0
Total Charges,0
Total Refunds,0
Total Extra Data Charges,0


In [45]:
master_df.duplicated().sum()

np.int64(0)

####**Ensure Data is Cleaned**

In [46]:
print("Shape:", master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())
print("Missing values:", master_df.isnull().sum().sum())

# Check negative values in numeric columns
numeric_cols = master_df.select_dtypes(include="number").columns
print("Negative values:")
print(master_df[numeric_cols].lt(0).sum())

# Check constant columns
constant_cols = [
    col for col in master_df.columns
    if master_df[col].nunique() <= 1
]
print("Constant columns:", constant_cols)

Shape: (7043, 47)
Duplicate rows: 0
Missing values: 0
Negative values:
Age                                     0
Number of Dependents                    0
Number of Referrals                     0
Tenure in Months                        0
Avg Monthly Long Distance Charges       0
Avg Monthly GB Download                 0
Monthly Charge                          0
Total Charges                           0
Total Refunds                           0
Total Extra Data Charges                0
Total Long Distance Charges             0
Total Revenue                           0
Satisfaction Score                      0
Churn Value                             0
Churn Score                             0
CLTV                                    0
Zip Code                                0
Latitude                                0
Longitude                            7043
Population                              0
dtype: int64
Constant columns: []


##**Save Updated Dataset**

In [49]:
# import os

# os.makedirs("/content/Telecom-Customer-Churn-Prediction/data", exist_ok=True)

# master_df.to_csv(
#     "/content/Telecom-Customer-Churn-Prediction/data/master_df.csv",
#     index=False
# )

import os

# Define your target folder and filename
folder_path = '/content/drive/MyDrive/ADSProject/Project_Root/Merged_Dataset'
file_name = 'master_df.csv'
full_path = os.path.join(folder_path, file_name)


# Save the DataFrame to Google Drive
master_df.to_csv(full_path, index=False)

In [50]:
print(os.path.exists(
    "/content/drive/MyDrive/ADSProject/Project_Root/Merged_Dataset/master_df.csv"
))

True
